In [1]:
import pandas as pd
import torch
import numpy as np
from tqdm import tqdm
from transformers import DistilBertTokenizer, DistilBertModel

W0516 00:53:20.396000 20276 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [ ]:
file_input = 'dataset_koreksi.csv'

print(f"Memuat data dari: {file_input}")
df = pd.read_csv(file_input)

df = df.dropna(subset=['clean_text']).reset_index(drop=True)

print(f"Dataset berhasil dimuat. Total baris: {len(df)}")

Memuat data dari: dataset_koreksi.csv
Dataset berhasil dimuat. Total baris: 8943


In [ ]:
model_name = 'cahya/distilbert-base-indonesian'

print("Mengunduh dan memuat Tokenizer dan Model DistilBERT")
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertModel.from_pretrained(model_name)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Model berhasil dimuat dan menggunakan perangkat: {device}")

Mengunduh dan memuat Tokenizer dan Model DistilBERT
Model berhasil dimuat dan menggunakan perangkat: cuda


In [4]:
teks_sampel = df['clean_text'].iloc[3179]
print(f"Teks Asli: '{teks_sampel}'")

Teks Asli: 'bangkai lu jangan ngehoax mulu pada takut enggak makan lu ya'


In [5]:
tokens = tokenizer(teks_sampel, return_tensors='pt', padding='max_length', truncation=True, max_length=128)
input_ids = tokens['input_ids'].to(device)
attention_mask = tokens['attention_mask'].to(device)

print(f"Ukuran Input IDs: {input_ids.shape} -> [Jumlah Batch, Panjang Token]")

with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

sequence_state = outputs.last_hidden_state
print(f"Ukuran Matriks Ekstraksi: {sequence_state.shape} -> [Jumlah Batch, Panjang Sekuens, Dimensi Fitur]")

Ukuran Input IDs: torch.Size([1, 128]) -> [Jumlah Batch, Panjang Token]
Ukuran Matriks Ekstraksi: torch.Size([1, 128, 768]) -> [Jumlah Batch, Panjang Sekuens, Dimensi Fitur]


In [ ]:
def extract_features_for_bilstm(text_list, batch_size=16):
    all_embeddings = []

    for i in tqdm(range(0, len(text_list), batch_size), desc="Proses Sequence Embedding"):
        batch_texts = text_list[i : i + batch_size]

        encoded_input = tokenizer(
            batch_texts,
            padding='max_length',
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )

        batch_input_ids = encoded_input['input_ids'].to(device)
        batch_attention_mask = encoded_input['attention_mask'].to(device)

        with torch.no_grad():
            model_output = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask)

        # Menyimpan matriks 3D ke dalam list
        batch_sequence_embeddings = model_output.last_hidden_state
        all_embeddings.append(batch_sequence_embeddings.cpu().numpy())

    return np.concatenate(all_embeddings, axis=0)

In [ ]:
list_teks = df['clean_text'].tolist()
matriks_x = extract_features_for_bilstm(list_teks, batch_size=16)

print("\nDimensi Matriks Fitur Keseluruhan (X, Y, Z):")
print(f"{matriks_x.shape} -> [Jumlah Teks, Panjang Sekuens, Dimensi DistilBERT]")

Proses Sequence Embedding: 100%|██████████| 559/559 [00:57<00:00,  9.75it/s]



Dimensi Matriks Fitur Keseluruhan (X, Y, Z):
(8943, 128, 768) -> [Jumlah Teks, Panjang Sekuens, Dimensi DistilBERT]


In [ ]:
lokasi_x = 'distilbert_sequence_embeddings.npy'
np.save(lokasi_x, matriks_x)
print(f"\nMatriks fitur (X) berhasil disimpan: {lokasi_x}")

array_y = df['label'].values
lokasi_y = 'label_sentimen.npy'
np.save(lokasi_y, array_y)
print(f"Array target (Y) berhasil disimpan: {lokasi_y}")


Matriks fitur (X) berhasil disimpan: distilbert_sequence_embeddings.npy
Array target (Y) berhasil disimpan: label_sentimen.npy
